# SLM vs flagship LLMs: financial report QA (TAT-QA)

Fine-tune **Qwen3-4B-Instruct** on TAT-QA on Kaggle **2x T4**, then compare it with flagship API models
on the same test set, the same output contract and the official scorer: accuracy, latency and cost.

**Before running:** Settings -> Accelerator **GPU T4 x2**, Internet **on**.
For the API baselines, add Kaggle Secrets (Add-ons -> Secrets): `ANTHROPIC_API_KEY` and/or `OPENAI_API_KEY`, `GEMINI_API_KEY`.
If the code is in a private GitHub repo, add `GITHUB_TOKEN` (read-only). Optional `HF_TOKEN` to push checkpoints to the Hugging Face Hub (protects a long run against the 12 h session limit).

Recommended: run the full training with **Save Version -> Save & Run All (Commit)** so it keeps running in the background.

## 1. Get the code

In [ ]:
import os, shutil, subprocess, base64
os.chdir("/kaggle/working")
REPO_URL = ""   # e.g. "https://github.com/<you>/FINETUNINGSLM.git" -- or leave empty and attach the code as a Kaggle Dataset
CODE_DATASET = "/kaggle/input/finetuningslm"   # where the attached dataset with src/ lives

if REPO_URL:
    if not os.path.exists("FINETUNINGSLM"):
        cmd = ["git"]
        try:  # private repo: read-only GitHub token from the GITHUB_TOKEN secret
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            # passed per command, so the token is never written to .git/config or the notebook output
            cmd += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]
        except Exception:
            print("no GITHUB_TOKEN secret - cloning as a public repo")
        result = subprocess.run(cmd + ["clone", "--quiet", REPO_URL, "FINETUNINGSLM"], capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError("git clone failed - check REPO_URL and that the GITHUB_TOKEN secret is attached "
                               "and can read this repository")
else:
    shutil.copytree(CODE_DATASET, "FINETUNINGSLM", dirs_exist_ok=True)
os.chdir("/kaggle/working/FINETUNINGSLM")
print(os.listdir("src"))

## 2. Install training libraries

In [ ]:
!pip install -q -U "transformers>=4.51" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.43" anthropic openai
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Secrets -> environment variables (only the ones you added)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for k in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY", "HF_TOKEN"]:
        try:
            os.environ[k] = secrets.get_secret(k)
            print("loaded", k)
        except Exception:
            pass
except ImportError:
    print("not on Kaggle")

## 3. Data: TAT-QA (train 13k / dev 1.7k / test 1.7k with official gold answers)

In [ ]:
!python src/prepare_data.py --out data

## 4. Fine-tune (QLoRA, fp16, both T4s)

Dry run first (~5 min): catches memory or version problems before committing hours.

In [ ]:
!python src/train.py --data data --out outputs/dryrun --max_train_samples 64 --val_samples 16     --epochs 1 --save_steps 1000 --batch_size 2 --grad_accum 2

Full run: ~12M tokens per epoch. Expect very roughly **2.5-3.5 h per epoch on 2x T4** (2 epochs by default).
If you hit out-of-memory, use `--batch_size 1 --grad_accum 16`.

**Resume is built in.** `--max_hours 10.5` stops training cleanly before Kaggle's 12 h limit, saving a checkpoint.
With `HUB_REPO` set (and the `HF_TOKEN` secret), every checkpoint is also pushed to a private Hugging Face repo.
To continue in a new session, just run the notebook again: `--resume` picks up the newest checkpoint, from local disk
or else from the Hub. Without the Hub, attach the previous version's output as input and pass
`--resume_from /kaggle/input/<notebook>/outputs/qwen3-4b-tatqa/checkpoint-<N>`.

In [ ]:
HUB_REPO = ""   # e.g. "your-hf-username/qwen3-4b-tatqa" (needs the HF_TOKEN secret); empty = local checkpoints only
hub = f"--hub_repo {HUB_REPO}" if HUB_REPO else ""
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 src/train.py     --data data --out outputs/qwen3-4b-tatqa --epochs 2 --max_hours 10.5 --resume {hub}

In [ ]:
# Runs only once training has finished (an interrupted run has no final_adapter yet -> re-run the notebook to resume)
![ -d outputs/qwen3-4b-tatqa/final_adapter ] && python src/merge_lora.py --adapter outputs/qwen3-4b-tatqa/final_adapter --out outputs/qwen3-4b-tatqa-merged || echo "training not finished yet - re-run to resume"

## 5. Local inference + benchmarking

vLLM gives realistic throughput numbers. If it fails to install or run on T4, add `--backend hf` to the commands below.
Each run is timed on **one T4** (cost = $0.35/GPU-hour by default; change with `--gpu_hourly_usd`).

In [ ]:
!pip install -q vllm

In [ ]:
# Hero: fine-tuned SLM, plain prompt it was trained on
![ -d outputs/qwen3-4b-tatqa-merged ] && python src/infer_local.py --model outputs/qwen3-4b-tatqa-merged --run slm_finetuned --mode finetuned || echo "no merged model yet"

In [ ]:
# Same base model before fine-tuning, given its best shot (4 worked examples + step-by-step reasoning)
!python src/infer_local.py --model Qwen/Qwen3-4B-Instruct-2507 --run slm_base_fewshot --mode fewshot

## 6. Flagship API baselines (same few-shot prompt, reasoning allowed, same calculator)

`--limit 500` evaluates a fixed, deterministic 500-question subset to cap spend; the report's *common view* compares every run
on exactly those questions. 
**Resumable:** re-running a command skips finished questions and retries failed ones. These API runs need no GPU, so you can also run them on your own computer and copy `results/preds/` back.
Rough Claude Opus 5.5 cost for 500 questions: ~$15-30 (depends on thinking length; the report shows the measured cost).

In [ ]:
# Runs only if the ANTHROPIC_API_KEY secret is attached, so training-only commits never call (or bill) the API
if os.environ.get("ANTHROPIC_API_KEY"):
    !python src/infer_api.py --provider anthropic --model claude-opus-5-5 --run claude_opus_5_5 --limit 500
else:
    print("ANTHROPIC_API_KEY not set - skipping the Claude baseline")

In [ ]:
# Optional extra contestants -- fill in model ids and their current $/1M token prices
# !python src/infer_api.py --provider anthropic --model claude-haiku-5-5 --run claude_haiku_5_5 --limit 500
# !python src/infer_api.py --provider openai --model <model-id> --run openai_flagship --limit 500 --price_in X --price_out Y
# !python src/infer_api.py --provider gemini --model <model-id> --run gemini_flagship --limit 500 --price_in X --price_out Y

## 7. Results

In [ ]:
!python src/evaluate.py --preds_dir results/preds --out results --hero slm_finetuned
from IPython.display import Image, Markdown, display
display(Markdown(open("results/summary.md").read()))
for f in ["results/accuracy.png", "results/accuracy_vs_cost.png"]:
    if os.path.exists(f):
        display(Image(f))

In [ ]:
# Look at mistakes: where does the fine-tuned model lose to the flagship (and vice versa)?
import json
def load(run): return {json.loads(l)["uid"]: json.loads(l) for l in open(f"results/preds/{run}.jsonl")}
slm, big = load("slm_finetuned"), load("claude_opus_5_5")
diff = [u for u in big if u in slm and (slm[u]["answer"], slm[u]["scale"]) != (big[u]["answer"], big[u]["scale"])]
print(len(diff), "questions where the two disagree")
for u in diff[:5]:
    print(u, "| SLM:", slm[u]["answer"], slm[u]["scale"], "| LLM:", big[u]["answer"], big[u]["scale"])